/workspace/KG-known0/260410_전체사이클_빈도상위10고정_gpt4_38개rel_IDPPA변형_entity

In [1]:
import pandas as pd
multi_predicate_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT/TACRED_full_binary_question_result.csv')
multi_predicate_df.head(2)

,id,relation,text,subject,subject_type,object,object_type,subj_start,subj_end,obj_start,...,profession,holds_advisory_position_for,occupation-title,is_a_creator,Title-Affiliation,author_of,expresses_affection_to,title or position held,has_profession,title or honorific of person
0,61b3aa9e36d5d8427e8e,org:alternate_names,"DC : Nov 30th thru Dec 2nd , Servicemembers Un...",Servicemembers United,ORGANIZATION,Servicemembers Legal Defense Network,ORGANIZATION,8,9,28,...,0,0,0,0,0,0,0,0,0,0
1,61b3aa9e368a0e3dfbd2,org:alternate_names,With the tally completed in more than half the...,All Basotho Convention,ORGANIZATION,ABC,ORGANIZATION,29,31,33,...,0,0,0,0,1,0,0,1,0,0


In [2]:
import spacy
import pandas as pd


nlp = spacy.load("en_core_web_sm")

multi_df = multi_predicate_df.copy()


multi_df["predicate"] = [[] for _ in range(len(multi_df))]


def find_entity_span(doc, entity_text):
    """
    doc 안에서 entity_text와 정확히 일치하는 연속 span을 찾아 반환
    없으면 None 반환
    """
    if not isinstance(entity_text, str) or not entity_text.strip():
        return None

    entity_text = entity_text.strip().lower()
    entity_words = entity_text.split()
    doc_words = [token.text.lower() for token in doc]

    n = len(entity_words)
    for i in range(len(doc_words) - n + 1):
        if doc_words[i:i+n] == entity_words:
            return doc[i:i+n]

    return None


def get_connected_verbs_from_span(span):
    """
    엔티티 span의 각 토큰으로부터 head 방향으로 올라가며
    처음 만나는 VERB/AUX를 predicate 후보로 수집
    """
    predicates = set()

    if span is None:
        return []

    for token in span:
        current = token
        visited = set()

        while current is not None and current.i not in visited:
            visited.add(current.i)

            if current.pos_ in {"VERB", "AUX"}:
                predicates.add(current.lemma_.lower())
                break

            if current.head == current:
                break

            current = current.head

    return list(predicates)


def extract_predicates_for_entities(text, subject, object_):
    """
    subject, object 각각과 관련된 술어를 추출
    결과: list[str]
    """
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)

    subj_span = find_entity_span(doc, subject)
    obj_span = find_entity_span(doc, object_)

    predicates = set()

    for p in get_connected_verbs_from_span(subj_span):
        predicates.add(p)

    for p in get_connected_verbs_from_span(obj_span):
        predicates.add(p)

    return list(predicates)



multi_df["predicate"] = multi_df.apply(
    lambda row: extract_predicates_for_entities(
        row["text"],
        row["subject"],
        row["object"]
    ),
    axis=1
)

In [3]:
multi_df.head()

,id,relation,text,subject,subject_type,object,object_type,subj_start,subj_end,obj_start,...,holds_advisory_position_for,occupation-title,is_a_creator,Title-Affiliation,author_of,expresses_affection_to,title or position held,has_profession,title or honorific of person,predicate
0,61b3aa9e36d5d8427e8e,org:alternate_names,"DC : Nov 30th thru Dec 2nd , Servicemembers Un...",Servicemembers United,ORGANIZATION,Servicemembers Legal Defense Network,ORGANIZATION,8,9,28,...,0,0,0,0,0,0,0,0,0,"[call, honor]"
1,61b3aa9e368a0e3dfbd2,org:alternate_names,With the tally completed in more than half the...,All Basotho Convention,ORGANIZATION,ABC,ORGANIZATION,29,31,33,...,0,0,0,1,0,0,1,0,0,"[trail, win]"
2,61b3aa9e36f31eacac2d,org:alternate_names,In 2009 after Fred has subscribed with Patriot...,Jacksonville,ORGANIZATION,Jacksonville,ORGANIZATION,9,9,34,...,0,0,0,0,0,0,0,0,0,[accept]
3,e7798a9e366634a95060,org:alternate_names,The banks ' core assets capital adequacy ratio...,China Banking Regulatory Commission,ORGANIZATION,CBRC,ORGANIZATION,21,24,26,...,0,0,1,1,1,0,1,0,0,[post]
4,e7798a9e365135c1cc36,org:alternate_names,"BAA , which is owned by a consortium headed by...",Global Infrastructure Partners,ORGANIZATION,U.S.-based,MISC,24,26,29,...,0,0,0,0,0,0,0,0,0,[sell]


In [4]:
multi_df.to_csv('/workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT/TACRED_full_binary_question_result_predicate.csv', index = False)

# predicate 통계

In [5]:
import ast

def count_predicate_items(x):
    
    if isinstance(x, list):
        return len(x)
    
    
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed)
        except Exception:
            pass
    
    
    return 0


multi_df["predicate_count"] = multi_df["predicate"].apply(count_predicate_items)


print(multi_df["predicate_count"].describe())


print(multi_df["predicate_count"].value_counts().sort_index())

count    1900.000000
mean        1.266842
std         0.536067
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
Name: predicate_count, dtype: float64
predicate_count
0      78
1    1245
2     570
3       6
4       1
Name: count, dtype: int64
